## Step 2 — extract and process major roads, pt 1
**# of cells in notebook:** 2

**Purpose:** `roads_1` contains the full drivable road network. From it, we want to extract ‘major roads’ and process them, converting dual carriage ways, roundabouts, interchanges, etc… into representative lines and junctions. When completed, the processed major roads will be one of the features, alongside linearized water features, the extent hull, and admin boundaries, used to create preliminary analysis zones across the city extent.  

**Input:**

- a geodatabase with:  `roads_1` and `extent_hull` layers.
- user needs to specify UTM zone in spatial reference section, cell 1

**Output:**

- final output is `roads_8`. However, the script outputs each intermediate processing layer so that it may be inspected for QA purposes. Note that the code will output an empty `roads_9` if there are no major roads to process.

**Main logic:**

1.	Project roads to UTM
2.	Extract major road features with SQL statements 
3.	Create several intermediate outputs, first cell creates up to `roads_3c_poly`.
4.	Cell 2 will linearize all polygons from `roads_3c_poly`, unless the user specifies otherwise. Inspect `roads_3c_poly` to determine whether you want to keep certain features as polygons. If so, pas these features’ `OBJECTIDs` into the `OBJECTIDs FROM roads_3c_poly TO REMOVE` section in cell 2. 
5.	Cell 2 creates up to `roads_8`, the tentative linearized major roads. Additional checks in the next notebook determine whether to process further. 

In [1]:
import arcpy
import os
import time

# -------------------------------------------------------------------
# ENVIRONMENT
# -------------------------------------------------------------------
arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False
arcpy.env.parallelProcessingFactor = "100"

# -------------------------------------------------------------------
# GEODATABASE
# -------------------------------------------------------------------
gdb = r"E:\World Bank deliverbale 1\_analysis\roads\roads.gdb"
arcpy.env.workspace = gdb

# -------------------------------------------------------------------
# SPATIAL REFERENCE
# -------------------------------------------------------------------
utm36n = arcpy.SpatialReference(32636)  # WGS 1984 UTM Zone 36N

# -------------------------------------------------------------------
# CORE INPUTS / OUTPUTS
# -------------------------------------------------------------------
roads_1 = os.path.join(gdb, "roads_1")
roads_1_tmp = os.path.join(gdb, "roads_1_tmp")
roads_1_backup_wgs84 = os.path.join(gdb, "roads_1_backup_wgs84")

roads_2 = os.path.join(gdb, "roads_2")
roads_2_backup = os.path.join(gdb, "roads_2_backup")
roads_2_ext = os.path.join(gdb, "roads_2_ext")
roads_2_mpsp = os.path.join(gdb, "roads_2_mpsp")

roads_3 = os.path.join(gdb, "roads_3")
roads_3a = os.path.join(gdb, "roads_3a")
roads_3b = os.path.join(gdb, "roads_3b")
roads_3c_poly = os.path.join(gdb, "roads_3c_poly")

roads_9 = os.path.join(gdb, "roads_9")

# -------------------------------------------------------------------
# GENERAL HELPERS
# -------------------------------------------------------------------
def delete_if_exists(path):
    if arcpy.Exists(path):
        arcpy.management.Delete(path)

def field_exists(fc, field_name):
    return field_name in [f.name for f in arcpy.ListFields(fc)]

def count_features(fc):
    return int(arcpy.management.GetCount(fc)[0])

def describe_fc(fc):
    d = arcpy.Describe(fc)
    sr_name = d.spatialReference.name if d.spatialReference else "Unknown"
    print(f"Dataset: {fc}")
    print(f"  Shape type: {d.shapeType}")
    print(f"  Spatial ref: {sr_name}")
    print(f"  Feature count: {count_features(fc)}")

def run_step(step_name, func, *args, **kwargs):
    print(f"\n--- Starting: {step_name} ---")
    t0 = time.time()
    result = func(*args, **kwargs)
    elapsed = time.time() - t0
    print(f"--- Finished: {step_name} in {elapsed:.1f} seconds ---")
    return result

# -------------------------------------------------------------------
# PREP STEPS
# -------------------------------------------------------------------
def project_roads_1_in_place():
    if not arcpy.Exists(roads_1):
        raise ValueError(f"roads_1 does not exist: {roads_1}")

    delete_if_exists(roads_1_tmp)
    delete_if_exists(roads_1_backup_wgs84)

    arcpy.management.CopyFeatures(roads_1, roads_1_backup_wgs84)
    print(f"Backup of original roads_1 created: {roads_1_backup_wgs84}")

    arcpy.management.Project(roads_1, roads_1_tmp, utm36n)
    print(f"Projected roads_1 to temporary UTM layer: {roads_1_tmp}")

    delete_if_exists(roads_1)
    arcpy.management.Rename(roads_1_tmp, "roads_1")
    print(f"Replaced original roads_1 with projected UTM version: {roads_1}")

    describe_fc(roads_1)

# -------------------------------------------------------------------
# MAIN WORKFLOW HELPERS
# -------------------------------------------------------------------
def select_and_export(input_fc, output_fc, sql_query, empty_output_fc):
    delete_if_exists("temp_layer")
    arcpy.management.MakeFeatureLayer(input_fc, "temp_layer", sql_query)

    selected_count = count_features("temp_layer")
    print(f"Selected feature count: {selected_count}")

    if selected_count == 0:
        delete_if_exists(empty_output_fc)
        arcpy.management.CopyFeatures("temp_layer", empty_output_fc)
        print(f"No features selected. Empty output written to {empty_output_fc}")
    else:
        delete_if_exists(output_fc)
        arcpy.management.CopyFeatures("temp_layer", output_fc)
        print(f"Selected records exported to {output_fc}")

    arcpy.management.Delete("temp_layer")

def add_field_and_calculate(input_fc, field_name, field_type, expression):
    if not field_exists(input_fc, field_name):
        arcpy.management.AddField(input_fc, field_name, field_type)
    arcpy.management.CalculateField(input_fc, field_name, expression, "PYTHON3")
    print(f"Field '{field_name}' added/calculated.")

def copy_features(input_fc, output_fc):
    delete_if_exists(output_fc)
    arcpy.management.CopyFeatures(input_fc, output_fc)
    print(f"Copied {input_fc} to {output_fc}")

def repair_geometry(input_fc):
    arcpy.management.RepairGeometry(input_fc)
    print(f"RepairGeometry applied to {input_fc}")

def extend_lines_in_place(input_fc, extend_length, extend_option):
    print(f"About to run ExtendLine on {input_fc}")
    print(f"  Extend length: {extend_length}")
    print(f"  Extend option: {extend_option}")
    print(f"  Feature count before extend: {count_features(input_fc)}")
    arcpy.edit.ExtendLine(input_fc, extend_length, extend_option)
    print(f"ExtendLine applied to {input_fc}")

def multipart_to_singlepart(input_fc, output_fc):
    delete_if_exists(output_fc)
    arcpy.management.MultipartToSinglepart(input_fc, output_fc)
    print(f"MultipartToSinglepart output: {output_fc}")

def merge_divided_roads(input_fc, output_fc, merge_field, merge_distance):
    delete_if_exists(output_fc)
    arcpy.cartography.MergeDividedRoads(input_fc, merge_field, merge_distance, output_fc)
    print(f"MergeDividedRoads output: {output_fc}")

def unsplit_lines(input_fc, output_fc):
    delete_if_exists(output_fc)
    arcpy.management.UnsplitLine(input_fc, output_fc)
    print(f"UnsplitLine output: {output_fc}")

def feature_to_polygon_and_select(input_fc, polygon_output_fc, final_output_fc):
    delete_if_exists(polygon_output_fc)
    delete_if_exists(final_output_fc)

    arcpy.management.FeatureToPolygon(input_fc, polygon_output_fc)
    print(f"FeatureToPolygon output: {polygon_output_fc}")

    if not field_exists(polygon_output_fc, "areaHA"):
        arcpy.management.AddField(polygon_output_fc, "areaHA", "DOUBLE")

    arcpy.management.CalculateGeometryAttributes(
        polygon_output_fc,
        [["areaHA", "AREA"]],
        area_unit="HECTARES"
    )

    delete_if_exists("temp_layer")
    arcpy.management.MakeFeatureLayer(polygon_output_fc, "temp_layer")
    arcpy.management.SelectLayerByAttribute("temp_layer", "NEW_SELECTION", '"areaHA" <= 7')

    selected_count = count_features("temp_layer")
    print(f"Polygon selection count (areaHA <= 7): {selected_count}")

    arcpy.management.CopyFeatures("temp_layer", final_output_fc)
    arcpy.management.Delete("temp_layer")

    print(f"Selected polygons with areaHA <= 7 exported to {final_output_fc}")

# -------------------------------------------------------------------
# SQL FOR roads_2
# -------------------------------------------------------------------
sql_query = """
(
    ("highway" LIKE '%motorway%' AND "highway" NOT LIKE '%motorway_link%' AND "highway" NOT LIKE '%trunk_link%' AND "highway" NOT LIKE '%primary_link%')
    OR
    ("highway" LIKE '%trunk%' AND "highway" NOT LIKE '%motorway_link%' AND "highway" NOT LIKE '%trunk_link%' AND "highway" NOT LIKE '%primary_link%')
    OR
    ("highway" LIKE '%primary%' AND "highway" NOT LIKE '%motorway_link%' AND "highway" NOT LIKE '%trunk_link%' AND "highway" NOT LIKE '%primary_link%')
)
OR
(
    "highway" LIKE '%motorway_link%' AND
    (
        "highway" LIKE '%motorway%' OR
        "highway" LIKE '%trunk%' OR
        "highway" LIKE '%primary%'
    ) AND
    "highway" <> 'motorway_link'
)
OR
(
    "highway" LIKE '%trunk_link%' AND
    (
        "highway" LIKE '%motorway%' OR
        "highway" LIKE '%trunk%' OR
        "highway" LIKE '%primary%'
    ) AND
    "highway" <> 'trunk_link'
)
OR
(
    "highway" LIKE '%primary_link%' AND
    (
        "highway" LIKE '%motorway%' OR
        "highway" LIKE '%trunk%' OR
        "highway" LIKE '%primary%'
    ) AND
    "highway" <> 'primary_link'
)
"""

# -------------------------------------------------------------------
# RUN SCRIPT A
# -------------------------------------------------------------------
if not arcpy.Exists(roads_1):
    raise ValueError(f"Input roads_1 does not exist: {roads_1}")

run_step("project roads_1 in place", project_roads_1_in_place)

run_step("select roads_2", select_and_export, roads_1, roads_2, sql_query, roads_9)

if not arcpy.Exists(roads_2):
    print("No roads_2 created. Stopping.")
else:
    describe_fc(roads_2)

    run_step("add maj_rd to roads_2", add_field_and_calculate, roads_2, "maj_rd", "SHORT", "1")

    run_step("backup roads_2", copy_features, roads_2, roads_2_backup)
    run_step("copy roads_2 to working layer roads_2_ext", copy_features, roads_2, roads_2_ext)
    run_step("repair geometry on roads_2_ext", repair_geometry, roads_2_ext)
    run_step("extend roads_2_ext", extend_lines_in_place, roads_2_ext, "400 Meters", "EXTENSION")

    describe_fc(roads_2_ext)

    run_step("multipart to singlepart", multipart_to_singlepart, roads_2_ext, roads_2_mpsp)
    describe_fc(roads_2_mpsp)

    run_step("merge divided roads", merge_divided_roads, roads_2_mpsp, roads_3, "maj_rd", "40 Meters")
    describe_fc(roads_3)

    run_step("unsplit roads_3", unsplit_lines, roads_3, roads_3a)
    describe_fc(roads_3a)

    run_step("feature to polygon and select", feature_to_polygon_and_select, roads_3a, roads_3b, roads_3c_poly)
    describe_fc(roads_3c_poly)

    print("\nDone with Script A.")
    print("Inspect roads_3c_poly in ArcGIS Pro.")
    print("Write down any OBJECTIDs that should be excluded before creating roads_3d_poly.")

ModuleNotFoundError: No module named 'arcpy'

In [ ]:
import arcpy
import os
import time

# -------------------------------------------------------------------
# ENVIRONMENT
# -------------------------------------------------------------------
arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False
arcpy.env.parallelProcessingFactor = "100"

# -------------------------------------------------------------------
# GEODATABASE
# -------------------------------------------------------------------
gdb = r"E:\World Bank deliverbale 1\_analysis\roads\roads.gdb"
arcpy.env.workspace = gdb

# -------------------------------------------------------------------
# USER INPUT: OBJECTIDs FROM roads_3c_poly TO REMOVE
# -------------------------------------------------------------------
# Example:
# objectids_to_remove = [12, 27, 31]
#
# If you do not want to remove anything, leave this as an empty list:
objectids_to_remove = [21]

# -------------------------------------------------------------------
# INPUTS / OUTPUTS
# -------------------------------------------------------------------
roads_3 = os.path.join(gdb, "roads_3")
roads_3c_poly = os.path.join(gdb, "roads_3c_poly")
roads_3c_poly_clean = os.path.join(gdb, "roads_3c_poly_clean")
roads_3d_poly = os.path.join(gdb, "roads_3d_poly")
roads_3e = os.path.join(gdb, "roads_3e")

roads_4 = os.path.join(gdb, "roads_4")
roads_5 = os.path.join(gdb, "roads_5")
roads_6 = os.path.join(gdb, "roads_6")
roads_7 = os.path.join(gdb, "roads_7")
roads_7_unsplit = os.path.join(gdb, "roads_7_unsplit")
roads_8 = os.path.join(gdb, "roads_8")

extent_hull = os.path.join(gdb, "extent_hull")

extent_lines = os.path.join(gdb, "extent_lines")
merged_lines = os.path.join(gdb, "merged_lines")

# -------------------------------------------------------------------
# GENERAL HELPERS
# -------------------------------------------------------------------
def delete_if_exists(path):
    if arcpy.Exists(path):
        arcpy.management.Delete(path)

def field_exists(fc, field_name):
    return field_name in [f.name for f in arcpy.ListFields(fc)]

def count_features(fc):
    return int(arcpy.management.GetCount(fc)[0])

def describe_fc(fc):
    d = arcpy.Describe(fc)
    sr_name = d.spatialReference.name if d.spatialReference else "Unknown"
    print(f"Dataset: {fc}")
    print(f"  Shape type: {d.shapeType}")
    print(f"  Spatial ref: {sr_name}")
    print(f"  Feature count: {count_features(fc)}")

def run_step(step_name, func, *args, **kwargs):
    print(f"\n--- Starting: {step_name} ---")
    t0 = time.time()
    result = func(*args, **kwargs)
    elapsed = time.time() - t0
    print(f"--- Finished: {step_name} in {elapsed:.1f} seconds ---")
    return result

# -------------------------------------------------------------------
# WORKFLOW HELPERS
# -------------------------------------------------------------------
def make_clean_roads_3c_poly():
    """
    Creates roads_3c_poly_clean from roads_3c_poly,
    excluding user-specified OBJECTIDs.
    """
    if not arcpy.Exists(roads_3c_poly):
        raise ValueError(f"Required input does not exist: {roads_3c_poly}")

    delete_if_exists(roads_3c_poly_clean)
    delete_if_exists("roads_3c_poly_layer")

    arcpy.management.MakeFeatureLayer(roads_3c_poly, "roads_3c_poly_layer")

    original_count = count_features("roads_3c_poly_layer")
    print(f"Original roads_3c_poly count: {original_count}")

    if objectids_to_remove:
        oid_field = arcpy.Describe(roads_3c_poly).OIDFieldName
        oid_list = ", ".join(str(oid) for oid in objectids_to_remove)

        # Select the OBJECTIDs to remove
        remove_sql = f"{arcpy.AddFieldDelimiters(gdb, oid_field)} IN ({oid_list})"
        arcpy.management.SelectLayerByAttribute(
            "roads_3c_poly_layer",
            "NEW_SELECTION",
            remove_sql
        )

        remove_count = count_features("roads_3c_poly_layer")
        print(f"Features selected for removal: {remove_count}")

        # Switch selection so that only retained polygons are copied
        arcpy.management.SelectLayerByAttribute(
            "roads_3c_poly_layer",
            "SWITCH_SELECTION"
        )

        retained_count = count_features("roads_3c_poly_layer")
        print(f"Features retained: {retained_count}")

        arcpy.management.CopyFeatures("roads_3c_poly_layer", roads_3c_poly_clean)

    else:
        print("No OBJECTIDs specified for removal. Copying all roads_3c_poly features.")
        arcpy.management.CopyFeatures(roads_3c_poly, roads_3c_poly_clean)

    arcpy.management.Delete("roads_3c_poly_layer")

    print(f"Clean polygon layer written to: {roads_3c_poly_clean}")
    describe_fc(roads_3c_poly_clean)

def dissolve_polygons(input_fc, output_fc):
    delete_if_exists(output_fc)

    # Clear any inherited tolerance/resolution settings before topology-building tools
    arcpy.ClearEnvironment("XYTolerance")
    arcpy.ClearEnvironment("XYResolution")

    print("XYTolerance before Dissolve:", arcpy.env.XYTolerance)
    print("XYResolution before Dissolve:", arcpy.env.XYResolution)

    arcpy.management.Dissolve(
        in_features=input_fc,
        out_feature_class=output_fc,
        dissolve_field="",
        statistics_fields="",
        multi_part="SINGLE_PART"
    )

    print(f"Dissolve output: {output_fc}")

def erase_features(input_fc, erase_fc, output_fc):
    delete_if_exists(output_fc)

    # Force-clear any inherited XY tolerance from the ArcGIS Pro / Notebook session
    arcpy.ClearEnvironment("XYTolerance")
    arcpy.ClearEnvironment("XYResolution")

    print("XYTolerance before Erase:", arcpy.env.XYTolerance)
    print("XYResolution before Erase:", arcpy.env.XYResolution)

    arcpy.analysis.Erase(
        in_features=input_fc,
        erase_features=erase_fc,
        out_feature_class=output_fc
    )

    print(f"Erase output: {output_fc}")

def collapse_hydro_polygon(input_fc, output_fc, connecting_features):
    delete_if_exists(output_fc)
    arcpy.cartography.CollapseHydroPolygon(
        input_fc,
        output_fc,
        merge_adjacent_input_polygons="MERGE_ADJACENT",
        connecting_features=connecting_features
    )
    print(f"CollapseHydroPolygon output: {output_fc}")

def merge_features(input_features, output_fc):
    delete_if_exists(output_fc)

    arcpy.ClearEnvironment("XYTolerance")
    arcpy.ClearEnvironment("XYResolution")

    arcpy.management.Merge(input_features, output_fc)

    print(f"Merge output: {output_fc}")
    
def calculate_geometry_and_delete(input_fc, output_fc):
    delete_if_exists(output_fc)

    if not field_exists(input_fc, "length"):
        arcpy.management.AddField(input_fc, "length", "DOUBLE")

    arcpy.management.CalculateGeometryAttributes(
        input_fc,
        [["length", "LENGTH"]],
        length_unit="METERS"
    )

    delete_if_exists("temp_layer")
    arcpy.management.MakeFeatureLayer(input_fc, "temp_layer")
    arcpy.management.SelectLayerByAttribute("temp_layer", "NEW_SELECTION", '"length" = 0')

    zero_count = count_features("temp_layer")
    print(f"Zero-length features to delete: {zero_count}")

    if zero_count > 0:
        arcpy.management.DeleteFeatures("temp_layer")

    arcpy.management.Delete("temp_layer")

    arcpy.management.CopyFeatures(input_fc, output_fc)
    print(f"Calculated geometry and removed zero-length features. Output: {output_fc}")

def extend_lines(input_fc, extend_length, extend_to):
    print(f"About to run final ExtendLine on {input_fc}")
    print(f"  Extend length: {extend_length}")
    print(f"  Extend option: {extend_to}")
    print(f"  Feature count before extend: {count_features(input_fc)}")

    arcpy.edit.ExtendLine(input_fc, extend_length, extend_to)

    print(f"ExtendLine applied to {input_fc}")

def unsplit_lines(input_fc, output_fc):
    delete_if_exists(output_fc)
    arcpy.management.UnsplitLine(input_fc, output_fc)
    print(f"UnsplitLine output: {output_fc}")

def extend_to_boundary(input_lines, extend_to_features, output_fc):
    try:
        delete_if_exists(extent_lines)
        delete_if_exists(merged_lines)
        delete_if_exists(output_fc)
        delete_if_exists("merged_layer")

        arcpy.management.FeatureToLine(extend_to_features, extent_lines, "", "NO_ATTRIBUTES")
        print(f"Converted boundary to lines: {extent_lines}")

        arcpy.management.Merge([input_lines, extent_lines], merged_lines)
        print(f"Merged lines: {merged_lines}")

        arcpy.management.MakeFeatureLayer(merged_lines, "merged_layer")

        arcpy.management.SelectLayerByLocation(
            "merged_layer",
            "INTERSECT",
            extent_lines,
            "",
            "NEW_SELECTION"
        )

        intersect_count = count_features("merged_layer")
        print(f"Features intersecting boundary lines: {intersect_count}")

        arcpy.management.SelectLayerByAttribute("merged_layer", "SWITCH_SELECTION")

        extend_count = count_features("merged_layer")
        print(f"Features to extend toward boundary: {extend_count}")

        arcpy.edit.ExtendLine("merged_layer", "200 Meters", "FEATURE")

        arcpy.management.SelectLayerByAttribute("merged_layer", "CLEAR_SELECTION")
        arcpy.management.SelectLayerByAttribute("merged_layer", "NEW_SELECTION")

        arcpy.management.SelectLayerByLocation(
            "merged_layer",
            "ARE_IDENTICAL_TO",
            extent_lines,
            "",
            "REMOVE_FROM_SELECTION"
        )

        final_count = count_features("merged_layer")
        print(f"Final line count to copy to roads_8: {final_count}")

        arcpy.management.CopyFeatures("merged_layer", output_fc)
        arcpy.management.Delete("merged_layer")

        print(f"Final output copied to {output_fc}")

    except arcpy.ExecuteError:
        print(arcpy.GetMessages(2))
        raise

# -------------------------------------------------------------------
# RUN SCRIPT B
# -------------------------------------------------------------------
if not arcpy.Exists(roads_3):
    raise ValueError(f"Required input roads_3 does not exist: {roads_3}")

if not arcpy.Exists(roads_3c_poly):
    raise ValueError(f"Required input roads_3c_poly does not exist: {roads_3c_poly}")

if not arcpy.Exists(extent_hull):
    raise ValueError(f"Required input extent_hull does not exist: {extent_hull}")

run_step("create cleaned roads_3c_poly", make_clean_roads_3c_poly)

run_step("dissolve cleaned polygons", dissolve_polygons, roads_3c_poly_clean, roads_3d_poly)
describe_fc(roads_3d_poly)

delete_if_exists(roads_4)
run_step("copy roads_3 to roads_4", arcpy.management.CopyFeatures, roads_3, roads_4)
describe_fc(roads_4)

# Important: use roads_3c_poly_clean here, not roads_3c_poly
run_step("erase roads_4 by roads_3c_poly_clean", erase_features, roads_4, roads_3c_poly_clean, roads_5)
describe_fc(roads_5)

run_step("collapse hydro polygon", collapse_hydro_polygon, roads_3d_poly, roads_3e, roads_5)
describe_fc(roads_3e)

run_step("merge roads_3e and roads_5", merge_features, [roads_3e, roads_5], roads_6)
describe_fc(roads_6)

run_step("calculate geometry and delete zero-length", calculate_geometry_and_delete, roads_6, roads_7)
describe_fc(roads_7)

run_step("final extend lines on roads_7", extend_lines, roads_7, "200 Meters", "FEATURE")
describe_fc(roads_7)

run_step("unsplit roads_7", unsplit_lines, roads_7, roads_7_unsplit)
describe_fc(roads_7_unsplit)

run_step("extend to boundary", extend_to_boundary, roads_7_unsplit, extent_hull, roads_8)
describe_fc(roads_8)

print("\nDone with Script B.")